<img src=images/gdd-logo.png width=300px align=right>

# Hyperparameter Tuning

In this notebook we are going to see how we can improve our models by looking at how we validate our models. In particular we will look at:

- [Cross validation](#cv) 
- [GridSearch with preprocessing steps](#gs-pre)
- [GridSearch with model parameters](#gs-hyper)
- [Putting it all together](#all-together)
- [More control over the hyperparameter search](#more-control)
- [RandomizedSearchCV](#randomized)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import ( OneHotEncoder,
                                    RobustScaler, 
                                    MinMaxScaler, 
                                    StandardScaler )
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC

In [ ]:
titanic_df = pd.read_csv('data/titanic.csv')

Below we use [the pipe method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html) to prep the data so that it is in the same state as in the previous notebook (drop unwanted columns and create our feature matrix $X$ and target vector $y$).

Pandas pipelines are a really nice method to organise your code. While this topic isn't covered in this notebook, below is an example of what you might want to aim for when writing in pandas.

In [ ]:
def drop_unwanted_cols(df, cols):
    return df.drop(columns=cols)


def create_Xy(df, target):
    df = df.reset_index(drop=True)
    X = df.drop(columns=[target])        
    y = df[target]
    return X, y

In [ ]:
unwanted_cols = ['embarked', 'sex', 'adult_male',
                 'deck', 'alive', 'class']

X, y = (
    titanic_df
    .pipe(drop_unwanted_cols, cols=unwanted_cols)
    .pipe(create_Xy, target='survived') 
)

In [ ]:
print(f'Original dataframe shape: {titanic_df.shape}')
print(f'Shape of X and y: {X.shape}, {y.shape}')

Notice that we have still have some missing values after this preprocessing.

In [ ]:
X.isna().sum()

This is because this time, we retained the age column. We shall have to take care of these missing values as part of our Scikit-Learn pipeline.

### Pipeline review

Remember our pipeline from earlier? We are going to use this as our initial set up of the `titanic_pipeline`.

In [ ]:
categorical_columns = ['who', 'embark_town', 'alone', 'pclass']

# One-hot encode
column_transformer = ColumnTransformer([
    ('one_hot_encoder', OneHotEncoder(drop='first'), categorical_columns),
    ], remainder="passthrough")

In [ ]:
# first we create our preprocessing step
preprocessor = Pipeline(steps=[
    ('column_transformer', column_transformer),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# add classifier and preprocessing to make the full pipeline
titanic_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', SVC())])

Notice we have set the imputing strategy to `median`. This means missing values in our *age* column will be replaced with the median age.

In [ ]:
X['age'].median()

## Validation

So far, we've been been applying a simple technique to validate our models. We've been using the `train_test_split` function so that we always withhold a portion of our data to test on. 

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=111, 
                                                    stratify=y)

However, the test set should **only be used to give a final evaluation of the model**. Otherwise we'll end up just fine-tuning our model to the test set (information leakage). This means our test set performance will not be a fair indication of how our model generalizes.

But what if, when we train our models, we want to compare the performance of multiple pipelines (e.g. to use `MinMaxScaler` versus `StandardScaler`) or using different hyperparameters? 

🔔 To be able to compare different alternatives we need to reserve an additional set of data, the **validation set**, to avoid leaking information from the test set into our decision making process.

 <img src="images/Hyperparameter_Tuning/validation.png" style="display: block;margin-left: auto;margin-right: auto;height: 100px"/>

<mark>**Question:** What would be a potential issue with reserving an additional split of the dataset?</mark>

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  When creating a dedicated validation split, less data can be used for training and testing, which can hurt the final performance of the model

</details>

<a id='cv'></a>

## Cross-Validation

🔔 Cross-Validation is a statistical method of evaluating and comparing models by dividing data into segments. We can use Cross-Validation when we don't have enough data to create a train, validation and test set. 

With Cross-Validation we split the data into multiple `k folds`. We then build the model on `k-1 folds` and validate the model on the left out fold:

 <img src="images/Hyperparameter_Tuning/crossvalidation.png" style="display: block;margin-left: auto;margin-right: auto;height: 300px"/>

There is a very practical way to perform Cross-Validation when creating our model. We can use `cross_val_score()` from `sklearn.model_selection` which returns an array of cross-validated scores for a model/pipeline on the data.

Let's see how our current model performs on the different validation splits:

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(titanic_pipeline,
                         X_train,
                         y_train,
                         cv=5,
                         scoring='accuracy'
                        )

print(f'Accuracy per fold: {scores.round(3)}')
print(f'Accuracy mean over folds: {scores.mean().round(3)}')
print(f'Accuracy var over folds: {scores.var().round(3)}')

Another way we might want to implement Cross-Validation is by determining the K-fold cross validation splits using `KFold()` from `sklearn.model_selection`. Then, for each split, the model can be fit and relevant metrics can be computed. 

## <mark>Exercise: Make changes to the parameters</mark>

Rerun the Cross-Validation but change some of the preprocessing parameters that we set in our pipeline. **Then answer the 2 questions below.**


Here are some options to try:

- Imputer strategy: `'mean'`, `'median'`, `'most_frequent'` or `'constant'`
- Scaler: `MinMaxScaler()`, `StandardScaler()` or `RobustScaler()`

### Questions:
1. With `strategy='constant'`, by default missing values will be filled with 0. Which parameter can you add to impute a different value?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  `fill_value`

</details>

2. Why would it be a bad idea to find the best combination by manually trying different combinations?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  The number of combinations grows exponentially with the number of hyperparameters. Manually finding the best combination is therefore extremely time consuming.

</details>

---
<a id='gs-preprocessing'></a>

## Grid Search with preprocessing steps

### Parameter tuning

🔔 To find the best parameters, we often want to check how each combination of parameters performs. This process is known as a **grid seach**. For each combination, we run cross-validation and save the scores. This will give us the best combination and the corresponding evaluation score. Finally, we retrain the model on 100% of the training data and evaluate on the test set.

| Pipeline performance                     | MinMax Scaler | Standard Scaler | Robust Scaler |
|------------------------------------------|---------------|-----------------|---------------|
| <b>Imputing strategy: Mean </b>          |               |                 |               |
| <b>Imputing strategy: Median </b>        |               |                 |               |
| <b>Imputing strategy: Most frequent </b> |               |                 |               |


 

In principle, we could execute the steps mentioned above manually, which would be a laborious task. Luckily, in Scikit-Learn they can be implemented directly.

### `GridSearchCV()`

`GridSearchCV()` from `sklearn.model_selection` allows tuning of model's parameters using Cross-Validated grid search. 

In [ ]:
from sklearn.model_selection import GridSearchCV

It requires a dictionary of parameters and the options it should compare. This does not only have to be model parameters, but parts of preprocessing too. We simply need to reference the correct pipeline step using 'step_name__parameter_name' as one of the dictionary keys (note the **double underscore** between the step and parameter name). In the end, the best set of parameters will be selected based on the provided metric.

In [ ]:
pd.DataFrame(
    titanic_pipeline.get_params().items(),
    columns=['parameter_name','parameter']
).loc[lambda df: df['parameter_name'].str.startswith('preprocessor') | df['parameter_name'].str.startswith('classifier')]

### Creating a grid parameters dictionary

To perform a `GridSearchCV()` we can create a dictionary where the keys are the parameter names as strings and the values are a collection of parameters on which to test. For example:

```python
{'preprocessor__scaler': [MinMaxScaler(), StandardScaler()]}
```

Pay attention to the parameter names: A string of the name of the model + 2 underscores + the specific parameter to search

This allows us to search possible parameters for any of the steps in the initial Pipeline (e.g. preprocessing too!) 

**Let's create our parameter dictionary!**

In [ ]:
preprocessing_parameters = {
    'preprocessor__imputer__strategy': ['constant', 'most_frequent'],
    'preprocessor__scaler': [MinMaxScaler(), StandardScaler()],
}

Now that we have a preprocessing_parameters dictionary we can pass this into a `GridSearchCV()` where `CV` stands for Cross-Validation. Here we perform over 3 folds and call our final output model (with pipeline) called `clf_svm`:

In [ ]:
clf_svm = GridSearchCV(titanic_pipeline, 
                       preprocessing_parameters, 
                       cv=3, 
                       scoring='accuracy')


clf_svm.fit(X_train, y_train)

print(f'CV accuracy score of the best SVM is: {clf_svm.best_score_:.3f}')
print(f'Best parameters were: {clf_svm.best_params_}')

### <mark>Exercise: Run `GridSearchCV()` on more parameters</mark>

Perform the following grid searches and find the `best_score` and `best_params` for each:

1. Add more parameters to `preprocessor__imputer__strategy` and `preprocessor__scaler`.

2. Add `preprocessor__imputer__fill_value` with 3 different parameters to search.

---
<a id='gs-params'></a>

## GridSearch with model hyperparameters

Each algorithm has different parameters which we can fine-tune to make the best model while testing how it will work on unseen data. **We will now be switching to using a** `RandomForestClassifier()`.

In a random forest a few parameters we might want to change are:

- `n_estimators`: Number of trees (`100`)
- `max_features`: Number of features considered at each split (`square root of total n features`)
- `criterion`: The method which each tree uses for its split (`gini`) [more info here](https://quantdare.com/decision-trees-gini-vs-entropy/)
- `max_depth`: The maximum depth of each tree (`None`)
- `min_samples_split`: The minimum number of samples required for a split (`2`)
- `min_samples_leaf`: The minimum number of samples required to be a lead node (`1`)

![](images/Hyperparameter_Tuning/tree-parameters.png)

**Let's perform a `GridSearchCV()` to see which combination of hyperparameters work best!**

We will need to create a new titanic_pipeline that has a `RandomForestClassifier()` as its classifier, instead of `SVC()`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

titanic_pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor),
                                   ('classifier', RandomForestClassifier())])

Now let's create a `model_parameters` dictionary using what we learned above.

In [ ]:
model_parameters = {'classifier__n_estimators': [40, 60, 90],
                    'classifier__max_depth': [2, 3, 5, 10],
                    'classifier__min_samples_leaf': [1,5,8]
                   }

Now that we have our dictionary let's perform our `GridSearchCV()` using the accuracy score as our measurement.

In [ ]:
grid_rf = GridSearchCV(titanic_pipeline_rf, 
                       model_parameters,
                       cv=3,
                       scoring='accuracy')

clf_rf = grid_rf.fit(X_train, y_train)

print(f'CV accuracy score of the best Random Forest is: {clf_rf.best_score_:.3f}')
print(f'Best parameters were: {clf_rf.best_params_}')

### <mark>Exercise: Fine-tune the Random Forest hyperparameters</mark>

Add some more hyperparameters to the `model_parameters` dictionary and check what the best score and best parameters are. 

Then answer the questions below.

### Questions:

1. Why does it take so much longer to run than when we were looking at the preprocessing steps?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  There are much more combinations to be evaluated.

</details>

2. Why is leaving the model hyperparameters as default not a good option when building a Random Forest?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  Random forests have several hyperparameters that can help to control underfitting/overfitting, such as the maximum depth and the minimum number of samples for a leaf node. By default, these hyperparameters are set to values that can lead to overfitting. Also, The number of hyperparameters for random forests is relatively large and therefore it is quite likely that the default configuration is not the optimal one.

</details>

---
<a id='all-together'></a>

### Putting it all together

You might be wondering, can we do this all together? I.e. can we perform one `GridSearchCV()` to find the best preprocessing steps as well as the best model parameters? The answer is yes! All we need to do is concatenate our dictionaries:

In [ ]:
all_parameters = {**model_parameters, **preprocessing_parameters}
all_parameters

Now we can run a `GridSearchCV()` over all these parameters in one go!

In [ ]:
grid_all = GridSearchCV(titanic_pipeline_rf, 
                        all_parameters, 
                        cv=3,
                        scoring='accuracy')

clf_all = grid_all.fit(X_train, y_train)

print(f'CV accuracy score of the best Random Forest is: {clf_all.best_score_:.3f}')
print(f'Best parameters were: {clf_all.best_params_}')
print(f'Best estimator was: {clf_all.best_estimator_}')

After completing the gridsearch, (by default) the pipeline is retrained on *the whole* dataset using the best parameters.

We can then use our test set to get a final evaluation of our model:

In [ ]:
clf_all.score(X_test,y_test)

It is usually also a good idea to not just blindly take the "best" model, but manually compare different models from the Grid search. Higher accuracy often comes at the price of over-fitting!

We can access the Cross-Validation results as a dataframe using:
```python 
pd.DataFrame(<cv_grid_object>.cv_results_)
```

In [ ]:
cv_results = pd.DataFrame(clf_all.cv_results_)
cv_results.sort_values('rank_test_score').head(6)

<details>
    
  <summary><span style="color:blue">Read about how to add different models to your hyperparameter search here. </span></summary>
  
## More control over the hyperparameter search

Let's now consider a scenario where you want to compare two different classifiers, a support vector machine and a random forest one.

Naively, you could create a dictionary like the following:

```python
all_parameters = {'classifier': [SVC(), RandomForestClassifier()],
                  'classifier__C': [.5, 1, 1.5], # SVC hyperparam
                  'classifier__kernel': ["linear", "poly", "rbf"], # SVC hyperparam
                  'classifier__n_estimators': [40, 60, 90], # RFC hyperparam
                  'classifier__max_depth': [2, 3, 5, 10], # RFC hyperparam
                  'classifier__min_samples_leaf': [1,5,8] # RFC hyperparam
                   }
```
However, if you use this dictionary with `GridSearchCV`, the search will explore all possible combinations of hyperparameters. For example, it will fit pipelines with a `RandomForestClassifier()` for every combination of the `C` and `kernel` hyperparameters. This would be a waste of resources since those hyperparameters are not related to the `RandomForestClassifier()`. 

To avoid this issue, `GridSearchCV` also accepts a _list_ of dictionaries as an input to give you more control over what combinations of hyperparameters are tested. It will only compare all possible combination of hyperparameters *within* each dictionary.


```python
svc_parameters = {'classifier': [SVC()],
                  'classifier__C': [.5, 1, 1.5],
                  'classifier__kernel': ["linear", "poly", "rbf"],
                   }
```

```python
rf_parameters = {'classifier': [RandomForestClassifier()],
                 'classifier__n_estimators': [40, 60, 90],
                 'classifier__max_depth': [2, 3, 5, 10],
                 'classifier__min_samples_leaf': [1,5,8]
                   }
```

```python
grid_all = GridSearchCV(pipeline, 
                        [svc_parameters, rf_parameters], 
                        scoring='accuracy')
```


</details>


<a id='randomized'></a>
## RandomizedSearchCV

If it is computationally unfeasible to compare all possible combinations of hyperparameters you want to try, you could use `RandomizedSearchCV` instead to `GridSearchCV`.

Instead of trying out all possible combination of hyperparameters, `RandomizedSearchCV` will randomly select some combination of hyperparameters to try. You can learn more about how to use in its [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html).

Random searches are not exhaustive and can be very greedy unless they run for a high number of iterations. In a lot of cases it makes sense to think about which hyperparameters are most relevant, and to try to reduce the size of the parameter space so that a `GridSearchCV` will complete in a feasible amount of time.


---
<img src='images/gdd-logo.png' align='right' width=200px>

## Conclusion

Validation can give us confidence that our models will work on unseen data (which will be the case when they go live!). 

We have covered a few steps to estimate how our model will perform:
* Cross Validation to estimate the skill of a machine learning model on unseen data
* GridSearch for preprocessing step selection
* GridSearch for model parameter tuning selection

These are advanced techniques that will take your modeling to the next level making sure that your models have longevity and perform consistently well.
<!-- 
### What's next?

The key to learning these skills is to adopt them. Have a project lined up at work? Great! Use the tools/techniques covered in these notebooks. Don't have a project? Head over to [kaggle](https://www.kaggle.com/) and find a dataset that interests you and apply these skills there!

When you're ready to continue learning, some more advanced machine learning techniques include: 
- Feature engineering
- Feature selection
- Building custom estimators (models) and transformations

These are covered in our [Advanced Data Science with Python](https://gdd.li/adswp) course! -->